In [1]:
import mne, pickle
import numpy as np
from moabb.datasets import Lee2019_MI
from moabb.paradigms import LeftRightImagery
from pathlib import Path

In [2]:
root = Path("fm_evaluation/data").resolve()
raw_dir = root / "raw"
resample = 128
out_dir = root / f"processed_8-30_{resample}"
label_map = {"left_hand": 0, "right_hand": 1}
phase_map = {"1train": "offline", "4test": "online"}
resample_points = 4*resample

In [3]:
mne.set_config("MNE_DATA", str(raw_dir))
mne.set_config("MNE_DATASETS_LEE2019_MI_PATH", str(raw_dir))

/tmp/ipykernel_4129804/3088876270.py:2: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_LEE2019_MI_PATH"
  mne.set_config("MNE_DATASETS_LEE2019_MI_PATH", str(raw_dir))


In [4]:
dataset = Lee2019_MI(train_run=True, test_run=False, resting_state=False,sessions=[1, 2])
paradigm = LeftRightImagery(fmin=8, fmax=30, tmin=0, tmax=4, resample=resample,)

In [5]:
def notch_filter_sub(dataset_sub):
    raw_data = dataset_sub.get_data()
    for sessions in raw_data.values():
        for runs in sessions.values():
            for raw in runs.values():
                raw.load_data()
                raw.notch_filter(freqs=[60], picks="eeg")
    dataset_sub._get_single_subject_data = lambda s: raw_data[s]
    return dataset_sub

In [7]:
for subject in range(1, 55):
    dataset_sub = Lee2019_MI(train_run=True, test_run=True, resting_state=False, sessions=[1, 2], subjects=[subject])
    dataset_sub = notch_filter_sub(dataset_sub)
    # get data for both train and test runs
    epochs, label, metadata = paradigm.get_data(dataset=dataset_sub, return_epochs=True)
    # V → μV
    X = (epochs.get_data(copy=True) * 1e6).astype(np.float32)
    label = np.asarray(label, dtype=str)
    metadata = metadata.reset_index(drop=True)
    # remain 4s × resample Hz = point
    assert np.isclose(epochs.info["sfreq"], resample)
    assert X.shape[-1] >= resample_points
    assert len(X) == len(label) == len(metadata)
    X = X[:, :, :resample_points]
    # convert label to int
    y = np.array([label_map[ll] for ll in label], dtype=np.int64)
    # save data for each session and run
    for (session, run), group in metadata.groupby(["session", "run"], sort=False):
        idx = group.index.to_numpy()
        phase = phase_map[run]
        data = {"X": X[idx], "label": label[idx], "y": y[idx],
            "metadata": group.reset_index(drop=True),
            "ch_names": list(epochs.ch_names),
            "sfreq": float(epochs.info["sfreq"]),
            "unit": "uV", "bandpass": [0.1, 75], "notch_hz": 60, 
            "reference": "original", "baseline": None,
            "tmin": 0, "duration": 4, "phase": phase,
        }
        filepath = out_dir / (f"sub-{subject:02d}_ses-{session}_{phase}.pkl")
        with open(filepath, "wb") as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Saved {filepath.name}: {data['X'].shape}", flush=True)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 6601 samples (6.601 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband e